In [3]:
"""
Environment Setup
"""

# Core libraries
import sys
import warnings
from pathlib import Path
import os

# Data science libraries
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: '%.2f' % x)
sns.set(style="whitegrid", palette="muted", font_scale=1.1)

# Plotly settings
px.defaults.template = "plotly_white" 
px.defaults.width = 1200
px.defaults.height = 700

# Constants
DATA_PATH = Path('../data/raw/data-kaggle')
PROCESSED_PATH = Path('../data/processed')
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
REPORTS_PATH = Path('../reports/profiles')
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

print("✅ Environment setup complete!")
print(f"📁 Raw data path: {DATA_PATH}")
print(f"📁 Processed data path: {PROCESSED_PATH}")
print(f"💾 Reports path: {REPORTS_PATH}")


✅ Environment setup complete!
📁 Raw data path: ../data/raw/data-kaggle
📁 Processed data path: ../data/processed
💾 Reports path: ../reports/profiles


In [4]:
def load_data(file_path):
    """
    Load data from a CSV file into a pandas DataFrame.

    Args:
        file_path (str): Path to the CSV file.

    Returns:
        pd.DataFrame: Loaded data.
    """
    try:
        data = pd.read_csv(file_path)
        print(f"Data loaded successfully from {file_path}")
        print(f"Shape: {data.shape}")
        print(f"Memory usage: {data.memory_usage(deep=True).sum() / (1024*1024):.2f} MB")
        return data
    except Exception as e:
        print(f"Error loading the data: {e}")
        return None

def clean_data(data):
    """
    Clean and preprocess the data following the approach from the Kaggle exploration notebook.

    Args:
        data (pd.DataFrame): Raw data to be cleaned.

    Returns:
        pd.DataFrame: Cleaned data.
        dict: Mapping of house types to integer values.
    """
    try:
        print("Cleaning and preprocessing data...")
        
        # Create a copy to avoid modifying the original
        data_copy = data.copy()
        
        # Replace non-numeric values in 'bath_num' and 'room_num' columns with 0 and convert to float
        if 'bath_num' in data_copy.columns:
            data_copy['bath_num'] = data_copy['bath_num'].replace('sin baños', '0').astype(float)
        
        if 'room_num' in data_copy.columns:
            data_copy['room_num'] = data_copy['room_num'].replace('sin habitación', '0').astype(float)

        # Convert 'garage' column to binary (0 if empty, 1 if not)
        if 'garage' in data_copy.columns:
            data_copy['garage'] = data_copy['garage'].notna().astype(int)

        # Identify unique values in 'house_type' and create a mapping to integers
        if 'house_type' in data_copy.columns:
            house_type_values = data_copy['house_type'].unique()
            house_type_mapping = {value: idx for idx, value in enumerate(house_type_values)}
            data_copy['house_type'] = data_copy['house_type'].map(house_type_mapping)
        else:
            house_type_mapping = {}

        # Drop unnecessary columns
        columns_to_drop = ['ground_size', 'kitchen', 'unfurnished', 'loc_street', 'ad_description']
        columns_to_drop = [col for col in columns_to_drop if col in data_copy.columns]
        if columns_to_drop:
            data_copy = data_copy.drop(columns=columns_to_drop)

        # Handle missing values by filling with median for specific columns
        for col in ['construct_date', 'm2_useful', 'lift']:
            if col in data_copy.columns and data_copy[col].dtype in [np.number]:
                data_copy[col].fillna(data_copy[col].median(), inplace=True)

        # Use one-hot encoding for categorical columns
        categorical_columns = [col for col in ['condition', 'heating', 'orientation'] 
                              if col in data_copy.columns]
        if categorical_columns:
            data_copy = pd.get_dummies(data_copy, columns=categorical_columns)

        print("Data cleaned successfully.")
        return data_copy, house_type_mapping
    except Exception as e:
        print(f"Error cleaning the data: {e}")
        return None, None

def split_data(data_cleaned, house_type_mapping):
    """
    Split the cleaned data into two datasets based on the 'house_type' column.

    Args:
        data_cleaned (pd.DataFrame): Cleaned data.
        house_type_mapping (dict): Mapping of house types to integer values.

    Returns:
        tuple: Two DataFrames, one for 'alquiler' and one for others.
    """
    try:
        # Identify codes corresponding to any 'alquiler' types in house_type_mapping
        alquiler_codes = [
            code for key, code in house_type_mapping.items()
            if 'alquiler' in str(key).lower()
        ]

        if alquiler_codes:
            # Split data based on 'house_type'
            alquiler_data = data_cleaned[data_cleaned['house_type'].isin(alquiler_codes)]
            other_data = data_cleaned[~data_cleaned['house_type'].isin(alquiler_codes)]
            print("Data split successfully.")
            print(f"Rental data: {alquiler_data.shape[0]} records")
            print(f"Sales data: {other_data.shape[0]} records")
            return alquiler_data, other_data
        else:
            print("No 'alquiler' types found in house_type.")
            return None, data_cleaned
    except Exception as e:
        print(f"Error splitting the data: {e}")
        return None, None

def save_data(data, output_file_path):
    """
    Save the cleaned data to a CSV file.

    Args:
        data (pd.DataFrame): Data to be saved.
        output_file_path (str): Path to save the CSV file.
    """
    try:
        data.to_csv(output_file_path, index=False)
        print(f"Data saved successfully to {output_file_path}.")
    except Exception as e:
        print(f"Error saving the data: {e}")

def dict_to_dataframe(dictionary, df_name='Mapping'):
    """
    Convert a dictionary to a pandas DataFrame.

    Args:
        dictionary (dict): Dictionary to convert.
        df_name (str): Name for the index of the DataFrame.

    Returns:
        pd.DataFrame: DataFrame representation of the dictionary.
    """
    df = pd.DataFrame(list(dictionary.items()), columns=['House_Type', 'Code'])
    df.index.name = df_name
    return df

# Load and process Álava housing data
alava_file = DATA_PATH / 'houses_alava.csv'
print(f"Processing data from: {alava_file}")

if alava_file.exists():
    # Load the data
    raw_data = load_data(alava_file)
    
    if raw_data is not None:
        # Clean the data
        cleaned_data, house_type_mapping = clean_data(raw_data)
        
        if cleaned_data is not None:
            # Split the data into rental and sales
            rental_data, sales_data = split_data(cleaned_data, house_type_mapping)
            
            # Save the processed data
            if rental_data is not None:
                save_data(rental_data, PROCESSED_PATH / 'houses_alava_cleaned_rent.csv')
            
            if sales_data is not None:
                save_data(sales_data, PROCESSED_PATH / 'houses_alava_cleaned_sale.csv')
            
            # Save the house type mapping
            if house_type_mapping:
                mapping_df = dict_to_dataframe(house_type_mapping, 'House_Type_Mapping')
                save_data(mapping_df, PROCESSED_PATH / 'houses_type_mapping.csv')
            
            # Set the variables for further analysis
            df_sales = sales_data
            df_rental_alava = rental_data
        else:
            print("❌ Failed to clean the data")
            df_sales = None
            df_rental_alava = None
    else:
        print("❌ Failed to load the data")
        df_sales = None
        df_rental_alava = None
else:
    print(f"❌ File not found: {alava_file}")
    df_sales = None
    df_rental_alava = None


Processing data from: ../data/raw/data-kaggle/houses_alava.csv
Data loaded successfully from ../data/raw/data-kaggle/houses_alava.csv
Shape: (3806, 36)
Memory usage: 6.61 MB
Cleaning and preprocessing data...
Data cleaned successfully.
Data split successfully.
Rental data: 127 records
Sales data: 3679 records
Data saved successfully to ../data/processed/houses_alava_cleaned_rent.csv.
Data saved successfully to ../data/processed/houses_alava_cleaned_sale.csv.
Data saved successfully to ../data/processed/houses_type_mapping.csv.


In [5]:
# Preview sales data
if df_sales is not None:
    print("\n=== SALES DATA PREVIEW ===")
    print("\nFirst 5 rows:")
    display(df_sales.head(5))
    
    print("\nColumn information:")
    for col, dtype in zip(df_sales.columns, df_sales.dtypes):
        print(f"- {col}: {dtype}")
    
    # Basic statistics
    print("\nBasic statistics:")
    display(df_sales.describe())

# Preview rental data
if df_rental_alava is not None:
    print("\n=== RENTAL DATA PREVIEW (ÁLAVA) ===")
    print("\nFirst 5 rows:")
    display(df_rental_alava.head(5))
    
    print("\nColumn information:")
    for col, dtype in zip(df_rental_alava.columns, df_rental_alava.dtypes):
        print(f"- {col}: {dtype}")
    
    # Basic statistics
    print("\nBasic statistics:")
    display(df_rental_alava.describe())



=== SALES DATA PREVIEW ===

First 5 rows:


,ad_last_update,air_conditioner,balcony,bath_num,built_in_wardrobe,chimney,construct_date,energetic_certif,floor,garage,...,"orientation_norte, oeste","orientation_norte, sur","orientation_norte, sur, este","orientation_norte, sur, este, oeste","orientation_norte, sur, oeste",orientation_oeste,orientation_sur,"orientation_sur, este","orientation_sur, este, oeste","orientation_sur, oeste"
0,Anuncio actualizado el 27 de marzo,0,0,2.00,0,0,1993.00,NaN,2 plantas,1,...,False,False,False,True,False,False,False,False,False,False
1,más de 5 meses sin actualizar,0,0,2.00,0,0,2006.00,no indicado,planta 2ª exterior,0,...,False,False,False,False,False,False,False,False,False,False
2,más de 5 meses sin actualizar,0,0,3.00,0,0,1993.00,no indicado,3 plantas,1,...,False,False,False,False,False,False,False,False,False,False
3,más de 5 meses sin actualizar,0,1,1.00,1,1,1993.00,en trámite,3 plantas,0,...,False,False,False,False,False,False,False,False,False,False
4,más de 5 meses sin actualizar,0,0,1.00,0,0,1993.00,no indicado,planta 1ª exterior,1,...,False,False,False,False,False,False,False,False,False,True



Column information:
- ad_last_update: object
- air_conditioner: int64
- balcony: int64
- bath_num: float64
- built_in_wardrobe: int64
- chimney: int64
- construct_date: float64
- energetic_certif: object
- floor: object
- garage: int64
- garden: int64
- house_id: int64
- house_type: int64
- lift: float64
- loc_city: object
- loc_district: object
- loc_full: object
- loc_neigh: object
- loc_zone: object
- m2_real: int64
- m2_useful: float64
- obtention_date: object
- price: int64
- reduced_mobility: int64
- room_num: float64
- storage_room: int64
- swimming_pool: int64
- terrace: int64
- condition_promoción de obra nueva: bool
- condition_segunda mano/buen estado: bool
- condition_segunda mano/para reformar: bool
- heating_calefacción central: bool
- heating_calefacción central: gas: bool
- heating_calefacción central: gasoil: bool
- heating_calefacción individual: bool
- heating_calefacción individual: bomba de frío/calor: bool
- heating_calefacción individual: eléctrica: bool
- heati

,air_conditioner,balcony,bath_num,built_in_wardrobe,chimney,construct_date,garage,garden,house_id,house_type,lift,m2_real,m2_useful,price,reduced_mobility,room_num,storage_room,swimming_pool,terrace
count,3679.00,3679.00,3679.00,3679.00,3679.00,3679.00,3679.00,3679.00,3679.00,3679.00,3679.00,3679.00,3679.00,3679.00,3679.00,3679.00,3679.00,3679.00,3679.00
mean,0.02,0.12,1.90,0.31,0.02,1990.10,0.45,0.26,60997534.39,3.21,0.84,456.46,108.08,244807.69,0.09,3.17,0.65,0.04,0.54
std,0.13,0.32,1.12,0.46,0.14,20.28,0.50,0.44,26341716.94,2.01,0.36,3931.06,80.77,163366.11,0.29,1.37,0.48,0.21,0.50
min,0.00,0.00,0.00,0.00,0.00,1700.00,0.00,0.00,299398.00,0.00,0.00,2.00,30.00,9300.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,1.00,0.00,0.00,1993.00,0.00,0.00,36519689.50,3.00,1.00,80.00,80.00,145000.00,0.00,2.00,0.00,0.00,0.00
50%,0.00,0.00,2.00,0.00,0.00,1993.00,0.00,0.00,81734822.00,3.00,1.00,100.00,87.00,199000.00,0.00,3.00,1.00,0.00,1.00
75%,0.00,0.00,2.00,1.00,0.00,1993.00,1.00,1.00,84065157.50,3.00,1.00,175.00,94.00,298000.00,0.00,4.00,1.00,0.00,1.00
max,1.00,1.00,34.00,1.00,1.00,2021.00,1.00,1.00,84861368.00,14.00,1.00,200000.00,2000.00,1700000.00,1.00,30.00,1.00,1.00,1.00



=== RENTAL DATA PREVIEW (ÁLAVA) ===

First 5 rows:


,ad_last_update,air_conditioner,balcony,bath_num,built_in_wardrobe,chimney,construct_date,energetic_certif,floor,garage,...,"orientation_norte, oeste","orientation_norte, sur","orientation_norte, sur, este","orientation_norte, sur, este, oeste","orientation_norte, sur, oeste",orientation_oeste,orientation_sur,"orientation_sur, este","orientation_sur, este, oeste","orientation_sur, oeste"
3627,Anuncio actualizado el 29 de marzo,0,0,1.00,0,0,1965.00,no indicado,planta 5ª exterior,0,...,False,False,False,False,False,False,False,False,False,False
3628,más de 2 meses sin actualizar,0,1,1.00,0,0,1993.00,NaN,planta 1ª exterior,0,...,False,False,False,False,False,True,False,False,False,False
3629,más de 2 meses sin actualizar,0,0,2.00,0,0,1993.00,no indicado,planta 1ª exterior,0,...,False,False,False,False,False,False,False,False,False,False
3630,más de 2 meses sin actualizar,0,0,2.00,0,0,1993.00,no indicado,planta 1ª exterior,0,...,False,False,False,False,False,False,True,False,False,False
3631,Anuncio actualizado el 21 de marzo,0,0,1.00,0,0,1993.00,no indicado,bajo exterior,0,...,False,False,False,False,False,False,False,False,False,False



Column information:
- ad_last_update: object
- air_conditioner: int64
- balcony: int64
- bath_num: float64
- built_in_wardrobe: int64
- chimney: int64
- construct_date: float64
- energetic_certif: object
- floor: object
- garage: int64
- garden: int64
- house_id: int64
- house_type: int64
- lift: float64
- loc_city: object
- loc_district: object
- loc_full: object
- loc_neigh: object
- loc_zone: object
- m2_real: int64
- m2_useful: float64
- obtention_date: object
- price: int64
- reduced_mobility: int64
- room_num: float64
- storage_room: int64
- swimming_pool: int64
- terrace: int64
- condition_promoción de obra nueva: bool
- condition_segunda mano/buen estado: bool
- condition_segunda mano/para reformar: bool
- heating_calefacción central: bool
- heating_calefacción central: gas: bool
- heating_calefacción central: gasoil: bool
- heating_calefacción individual: bool
- heating_calefacción individual: bomba de frío/calor: bool
- heating_calefacción individual: eléctrica: bool
- heati

,air_conditioner,balcony,bath_num,built_in_wardrobe,chimney,construct_date,garage,garden,house_id,house_type,lift,m2_real,m2_useful,price,reduced_mobility,room_num,storage_room,swimming_pool,terrace
count,127.00,127.00,127.00,127.00,127.00,127.00,127.00,127.00,127.00,127.00,127.00,127.00,127.00,127.00,127.00,127.00,127.00,127.00,127.00
mean,0.03,0.13,2.10,0.46,0.00,1992.33,0.35,0.21,71843025.62,16.06,0.91,174.61,109.44,1015.21,0.09,2.94,0.41,0.03,0.37
std,0.18,0.33,3.34,0.50,0.00,6.53,0.48,0.41,23330339.11,2.19,0.29,310.15,174.57,596.30,0.28,3.06,0.49,0.18,0.48
min,0.00,0.00,1.00,0.00,0.00,1960.00,0.00,0.00,1811730.00,15.00,0.00,35.00,39.00,350.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,1.00,0.00,0.00,1993.00,0.00,0.00,81764170.50,15.00,1.00,75.00,80.00,717.50,0.00,2.00,0.00,0.00,0.00
50%,0.00,0.00,2.00,0.00,0.00,1993.00,0.00,0.00,84180864.00,15.00,1.00,90.00,87.00,850.00,0.00,3.00,0.00,0.00,0.00
75%,0.00,0.00,2.00,1.00,0.00,1993.00,1.00,0.00,84675660.50,16.00,1.00,120.00,90.00,1100.00,0.00,3.00,1.00,0.00,1.00
max,1.00,1.00,38.00,1.00,0.00,2015.00,1.00,1.00,84861203.00,24.00,1.00,2000.00,2000.00,5000.00,1.00,35.00,1.00,1.00,1.00


In [ ]:
def generate_basic_profile(df, title):
    """
    Generate a basic profile report without external libraries
    
    Args:
        df: DataFrame to profile
        title: Title for the report
    
    Returns:
        Dictionary with basic statistics
    """
    print(f"Generating basic profile for: {title}")
    print(f"DataFrame shape: {df.shape}")
    
    # Basic statistics
    profile = {
        "shape": df.shape,
        "dtypes": df.dtypes,
        "missing_values": df.isna().sum(),
        "missing_percentage": (df.isna().sum() / len(df) * 100).round(2),
        "duplicates": df.duplicated().sum(),
    }
    
    # Numeric statistics
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    if len(numeric_cols) > 0:
        profile["numeric_stats"] = df[numeric_cols].describe()
    
    # Categorical statistics
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    if len(cat_cols) > 0:
        profile["categorical_stats"] = {
            col: {
                "unique_values": df[col].nunique(),
                "top_values": df[col].value_counts().head(5).to_dict()
            } for col in cat_cols
        }
    
    # Print summary
    print("\n=== PROFILE SUMMARY ===")
    print(f"Rows: {profile['shape'][0]}")
    print(f"Columns: {profile['shape'][1]}")
    print(f"Duplicate rows: {profile['duplicates']}")
    print("\nMissing values:")
    missing = profile['missing_values'][profile['missing_values'] > 0]
    if len(missing) > 0:
        for col, count in missing.items():
            print(f"  - {col}: {count} ({profile['missing_percentage'][col]}%)")
    else:
        print("  No missing values")
    
    return profile

def try_generate_profile_report(data, report_title, output_path=None):
    """
    Try to generate a profile report using ydata_profiling if available.
    Falls back to basic profile if not available.
    
    Args:
        data (pd.DataFrame): The dataset for which the profile report is to be generated.
        report_title (str): Title for the profile report.
        output_path (str, optional): Path to save the HTML report.
        
    Returns:
        Basic profile dictionary if ydata_profiling is not available
    """
    try:
        # Try to import ydata_profiling
        from ydata_profiling import ProfileReport
        
        print(f"Generating profile report for: {report_title}")
        
        # Generate the profile report
        profile = ProfileReport(
            data, 
            title=report_title,
            explorative=True,
            progress_bar=True
        )
        
        # Save the report to an HTML file if path provided
        if output_path:
            profile.to_file(output_path)
            print(f"Profile report saved to: {output_path}")
            
        return profile
    except ImportError:
        print("ydata_profiling not available. Using basic profile instead.")
        return generate_basic_profile(data, report_title)
    except Exception as e:
        print(f"Error generating profile report: {e}")
        return generate_basic_profile(data, report_title)


In [ ]:
# Generate profile for sales data
if df_sales is not None:
    print("Generating profile for sales data...")
    
    # Try to generate a comprehensive profile report
    sales_report_path = REPORTS_PATH / "alava_sales_profile.html"
    sales_profile = try_generate_profile_report(
        df_sales,
        "Álava Real Estate Sales",
        output_path=sales_report_path
    )
    
    # If it's a DataFrame (basic profile), display it
    if isinstance(sales_profile, dict) and "numeric_stats" in sales_profile:
        print("\nNumeric Statistics:")
        display(sales_profile["numeric_stats"])
    else:
        # If it's a ProfileReport object, display it
        display(sales_profile)
    
    # Show a sample of the data
    print("\nSample data:")
    display(df_sales.head())
else:
    print("❌ Cannot generate sales profile: No data available")


In [ ]:
# Generate profile for rental data
if df_rental_alava is not None:
    print("Generating profile for rental data...")
    
    # Try to generate a comprehensive profile report
    rental_report_path = REPORTS_PATH / "alava_rental_profile.html"
    rental_profile = try_generate_profile_report(
        df_rental_alava,
        "Álava Real Estate Rentals",
        output_path=rental_report_path
    )
    
    # If it's a DataFrame (basic profile), display it
    if isinstance(rental_profile, dict) and "numeric_stats" in rental_profile:
        print("\nNumeric Statistics:")
        display(rental_profile["numeric_stats"])
    else:
        # If it's a ProfileReport object, display it
        display(rental_profile)
    
    # Show a sample of the data
    print("\nSample data:")
    display(df_rental_alava.head())
else:
    print("❌ Cannot generate rental profile: No data available")


In [ ]:
if df_sales is not None and df_rental_alava is not None:
    # Prepare data for comparison
    # First, identify common metrics that can be compared
    
    # For sales, we typically have price in total amount
    # For rentals, we typically have price per month
    
    # Let's calculate price per square meter for both
    
    # Check if we have the necessary columns
    sales_has_required = all(col in df_sales.columns for col in ['price', 'surface'])
    rental_has_required = all(col in df_rental_alava.columns for col in ['price', 'surface'])
    
    if sales_has_required and rental_has_required:
        # Calculate price per square meter
        df_sales_with_metrics = df_sales.copy()
        df_sales_with_metrics['price_per_sqm'] = df_sales['price'] / df_sales['surface']
        
        df_rental_with_metrics = df_rental_alava.copy()
        df_rental_with_metrics['price_per_sqm'] = df_rental_alava['price'] / df_rental_alava['surface']
        df_rental_with_metrics['annual_price'] = df_rental_alava['price'] * 12  # Annualized rental price
        
        # Calculate summary statistics
        sales_metrics = {
            'count': len(df_sales_with_metrics),
            'avg_price': df_sales_with_metrics['price'].mean(),
            'median_price': df_sales_with_metrics['price'].median(),
            'avg_surface': df_sales_with_metrics['surface'].mean(),
            'avg_price_per_sqm': df_sales_with_metrics['price_per_sqm'].mean()
        }
        
        rental_metrics = {
            'count': len(df_rental_with_metrics),
            'avg_price': df_rental_with_metrics['price'].mean(),
            'median_price': df_rental_with_metrics['price'].median(),
            'avg_surface': df_rental_with_metrics['surface'].mean(),
            'avg_price_per_sqm': df_rental_with_metrics['price_per_sqm'].mean(),
            'avg_annual_price': df_rental_with_metrics['annual_price'].mean()
        }
        
        # Display comparison
        print("=== COMPARATIVE METRICS: SALES vs. RENTALS ===\n")
        print(f"Sales listings: {sales_metrics['count']:,}")
        print(f"Rental listings: {rental_metrics['count']:,}\n")
        
        print(f"Average sales price: €{sales_metrics['avg_price']:,.2f}")
        print(f"Average monthly rental: €{rental_metrics['avg_price']:,.2f}")
        print(f"Average annual rental: €{rental_metrics['avg_annual_price']:,.2f}\n")
        
        print(f"Average property size (sales): {sales_metrics['avg_surface']:,.2f} m²")
        print(f"Average property size (rentals): {rental_metrics['avg_surface']:,.2f} m²\n")
        
        print(f"Average price per m² (sales): €{sales_metrics['avg_price_per_sqm']:,.2f}")
        print(f"Average price per m² (rentals): €{rental_metrics['avg_price_per_sqm']:,.2f} /month")
        
        # Calculate yield (annual rental income / property price)
        avg_yield = rental_metrics['avg_annual_price'] / sales_metrics['avg_price'] * 100
        print(f"\nEstimated average rental yield: {avg_yield:.2f}%")
        
        # Create comparison visualizations
        
        # 1. Property size distribution comparison
        fig = go.Figure()
        
        # Add sales data
        fig.add_trace(go.Histogram(
            x=df_sales['surface'].dropna().values,
            name='Sales',
            opacity=0.7,
            nbinsx=30
        ))
        
        # Add rental data
        fig.add_trace(go.Histogram(
            x=df_rental_alava['surface'].dropna().values,
            name='Rentals',
            opacity=0.7,
            nbinsx=30
        ))
        
        fig.update_layout(
            title_text='Property Size Distribution: Sales vs. Rentals',
            xaxis_title_text='Surface Area (m²)',
            yaxis_title_text='Count',
            bargap=0.2,
            bargroupgap=0.1
        )
        
        fig.show()
        
        # 2. Price distribution comparison (need to normalize)
        fig = go.Figure()
        
        # For sales, we'll use price per sqm
        sales_price_per_sqm = df_sales_with_metrics['price_per_sqm'].dropna().values
        
        # For rentals, also price per sqm
        rental_price_per_sqm = df_rental_with_metrics['price_per_sqm'].dropna().values
        
        # Add sales data
        fig.add_trace(go.Histogram(
            x=sales_price_per_sqm,
            name='Sales (€/m²)',
            opacity=0.7,
            nbinsx=30
        ))
        
        # Add rental data
        fig.add_trace(go.Histogram(
            x=rental_price_per_sqm,
            name='Rentals (€/m²/month)',
            opacity=0.7,
            nbinsx=30
        ))
        
        fig.update_layout(
            title_text='Price per m² Distribution: Sales vs. Rentals',
            xaxis_title_text='Price per m²',
            yaxis_title_text='Count',
            bargap=0.2,
            bargroupgap=0.1
        )
        
        fig.show()
    else:
        print("❌ Cannot compare metrics: Missing required columns (price and/or surface)")
else:
    print("❌ Cannot compare markets: Missing data for either sales or rentals")


In [ ]:
"""
Summarize key findings and outline next steps for modeling
"""

# Key findings from the analysis
print("=== KEY FINDINGS ===\n")
print("1. Data Quality and Structure:")
print("   - [Will be filled after running the analysis]")
print("   - [Will be filled after running the analysis]")
print("\n2. Market Characteristics:")
print("   - [Will be filled after running the analysis]")
print("   - [Will be filled after running the analysis]")
print("\n3. Comparative Analysis:")
print("   - [Will be filled after running the analysis]")
print("   - [Will be filled after running the analysis]")

print("\n=== NEXT STEPS ===\n")
print("1. Data Preprocessing:")
print("   - Handle missing values identified in the profile reports")
print("   - Address outliers in price and surface area")
print("   - Normalize or standardize numerical features")
print("   - Encode categorical variables appropriately")

print("\n2. Feature Engineering:")
print("   - Create location-based features (distance to amenities, public transport)")
print("   - Extract temporal features from date fields if available")
print("   - Generate interaction terms between key variables")
print("   - Calculate price indices and trends over time")

print("\n3. Modeling Approach:")
print("   - Develop separate models for sales and rental predictions")
print("   - Consider ensemble methods combining multiple algorithms")
print("   - Implement time series forecasting for price trends")
print("   - Evaluate models using appropriate cross-validation strategies")

print("\n4. Validation and Interpretation:")
print("   - Compare model performance across different property types")
print("   - Analyze feature importance to understand market drivers")
print("   - Validate predictions against historical trends")
print("   - Interpret results in the context of economic indicators")
